# 02 · Rescore & Iterate on Tolerances and Gates

Inspect a saved run directory (`EvalQuestion.jsonl`, `EvalAsk.jsonl`, `FabricIqQuery.jsonl`), re-run deterministic scoring without re-calling Copilot Chat, and compare pass rates across model iterations. Defaults safely to `MOCK = True` when credentials are absent.

In [ ]:
import pathlib
import sys

repo = pathlib.Path.cwd()
while repo != repo.parent and not (repo / "src" / "cfeval").exists():
    repo = repo.parent
if (repo / "src" / "cfeval").exists() and str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))

from cfeval.cli import build_harness
from cfeval.config import Settings
from cfeval.pipeline import load_run, run_metrics
from cfeval.scoring import deterministic_checks, result_row
from cfeval.sinks import JsonlSink

MOCK = True
out_root = repo / "build" / "notebook-runs"
settings = Settings.load(env_file=None, output_dir=str(out_root))

In [ ]:
# Capture (or reuse) an offline baseline run folder
baseline_outcome = build_harness(
    settings, mock=MOCK, publish=False, profile="baseline", seed=11,
    run_id="nb02-captured-run", run_name="Captured Run for Offline Rescoring",
).run()

questions, asks = load_run(baseline_outcome.folder)
sink = JsonlSink(baseline_outcome.folder)
queries = sink.read("FabricIqQuery")
print(f"Loaded {len(questions)} questions, {len(asks)} asks, and {len(queries)} DAX trace rows from {baseline_outcome.folder.name}")

In [ ]:
# Re-score captured asks locally with relaxed numeric tolerance (zero network calls)
q_by_id = {q.id: q for q in questions}
queries_by_ask: dict[str, list[dict]] = {}
for qrow in queries:
    queries_by_ask.setdefault(qrow["AskId"], []).append(qrow)

rescored_rows = []
for ask in asks:
    q = q_by_id[ask["QuestionId"]]
    q_relaxed = type(q)(**{**q.__dict__, "rel_tol": 0.02})
    ask_queries = queries_by_ask.get(ask["AskId"], [])
    models = sorted({r.get("ItemName") for r in ask_queries if r.get("ItemName")})
    trace = {
        "status": "observed" if ask_queries else "pending",
        "models": models,
        "stats": {"count": len(ask_queries), "total": sum(float(r.get("DurationMs") or 0) for r in ask_queries), "max": 0.0, "first_offset": None},
    }
    artifacts = ask.get("Artifacts") or []
    checks = deterministic_checks(q_relaxed, ask, artifacts, trace)
    rescored_rows.append(
        result_row("nb02-rescored", q_relaxed, ask, artifacts, trace, checks, "fabric-native", "2026-11-05T00:00:00Z")
    )

print("Original metrics:", baseline_outcome.metrics)
print("Rescored metrics (rel_tol=2%):", run_metrics(rescored_rows))